# Victorian Rental Rights RAG Evaluation Pipeline

This notebook runs the Victorian Rental Rights RAG system through a structured evaluation pipeline.

The aim is to evaluate the system at each stage rather than only judging the final answer. This includes retrieval performance, answer quality, faithfulness, citation behaviour and Out-of-KB handling.

The same test collection will be reused across later system versions so improvements can be measured consistently.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import math
import requests

from rank_bm25 import BM25Okapi

# Pipeline configuration
SYSTEM_VERSION = "V1"
OLLAMA_MODEL = "llama3.2:3b"
OLLAMA_URL = "http://localhost:11434/api/generate"
TOP_K = 5

# Locate processed project data
CURRENT_DIR = Path.cwd()

if (CURRENT_DIR / "data" / "processed").exists():
    DATA_DIR = CURRENT_DIR / "data" / "processed"
elif (CURRENT_DIR.parent / "data" / "processed").exists():
    DATA_DIR = CURRENT_DIR.parent / "data" / "processed"
else:
    raise FileNotFoundError("Could not locate data/processed folder")

kb_df = pd.read_json(
    DATA_DIR / "rental_kb_chunks.jsonl",
    lines=True
)

test_df = pd.read_json(
    DATA_DIR / "rental_test_collection_v1.jsonl",
    lines=True
)

print(f"System version: {SYSTEM_VERSION}")
print(f"Model: {OLLAMA_MODEL}")
print(f"Top-k: {TOP_K}")
print(f"Knowledge-base chunks: {len(kb_df)}")
print(f"Test questions: {len(test_df)}")

System version: V1
Model: llama3.2:3b
Top-k: 5
Knowledge-base chunks: 32
Test questions: 15


## 1. Retrieval Pipeline

Build the BM25 retriever and retrieve the top-k knowledge-base chunks for every test question.

Retrieval metrics are calculated for Known and Inferred questions where ground-truth relevant chunks are available. Out-of-KB questions are still passed through retrieval, but are excluded from retrieval relevance metrics because they intentionally have no relevant chunks.

In [2]:
def tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())


# Build BM25 index
tokenized_corpus = kb_df["text"].apply(tokenize).tolist()
bm25 = BM25Okapi(tokenized_corpus)


def retrieve_bm25(question, top_k=TOP_K):
    query_tokens = tokenize(question)
    scores = bm25.get_scores(query_tokens)

    results = kb_df.copy()
    results["bm25_score"] = scores

    return (
        results
        .sort_values("bm25_score", ascending=False)
        .head(top_k)
        .reset_index(drop=True)
    )


def evaluate_retrieval(row, top_k=TOP_K):
    results = retrieve_bm25(row["question"], top_k)

    retrieved_ids = results["chunk_id"].tolist()
    retrieved_scores = results["bm25_score"].tolist()
    retrieved_contexts = results["text"].tolist()

    relevant_ids = set(row["relevant_chunk_ids"])

    # Out-of-KB questions have no relevant chunks
    if not relevant_ids:
        return pd.Series({
            "retrieved_chunk_ids": retrieved_ids,
            "retrieved_scores": retrieved_scores,
            "retrieved_contexts": retrieved_contexts,
            "first_relevant_rank": np.nan,
            "recall_at_k": np.nan,
            "mrr_at_k": np.nan,
            "ndcg_at_k": np.nan
        })

    matched = [
        chunk_id for chunk_id in retrieved_ids
        if chunk_id in relevant_ids
    ]

    recall = len(matched) / len(relevant_ids)

    first_rank = next(
        (
            rank
            for rank, chunk_id in enumerate(retrieved_ids, start=1)
            if chunk_id in relevant_ids
        ),
        None
    )

    mrr = 1 / first_rank if first_rank else 0

    dcg = sum(
        1 / math.log2(rank + 1)
        for rank, chunk_id in enumerate(retrieved_ids, start=1)
        if chunk_id in relevant_ids
    )

    ideal_relevant = min(len(relevant_ids), top_k)

    idcg = sum(
        1 / math.log2(rank + 1)
        for rank in range(1, ideal_relevant + 1)
    )

    ndcg = dcg / idcg if idcg else 0

    return pd.Series({
        "retrieved_chunk_ids": retrieved_ids,
        "retrieved_scores": retrieved_scores,
        "retrieved_contexts": retrieved_contexts,
        "first_relevant_rank": first_rank,
        "recall_at_k": recall,
        "mrr_at_k": mrr,
        "ndcg_at_k": ndcg
    })


retrieval_results = test_df.apply(
    evaluate_retrieval,
    axis=1
)

results_df = pd.concat(
    [test_df.copy(), retrieval_results],
    axis=1
)

display(
    results_df[
        [
            "question_id",
            "question_type",
            "retrieved_chunk_ids",
            "first_relevant_rank",
            "recall_at_k",
            "mrr_at_k",
            "ndcg_at_k"
        ]
    ]
)

,question_id,question_type,retrieved_chunk_ids,first_relevant_rank,recall_at_k,mrr_at_k,ndcg_at_k
0,K01,Known,"[RG_P35, RG_P38, RG_P04, RG_P23, RG_P07]",5.0,1.00,0.2,0.386853
1,K02,Known,"[RG_P08, RG_P21, MS_P01, RG_P12, RG_P13]",1.0,1.00,1.0,1.000000
2,K03,Known,"[RG_P13, MS_P01, RG_P39, RG_P17, RG_P04]",1.0,1.00,1.0,1.000000
3,K04,Known,"[RG_P23, RG_P17, RG_P19, RG_P34, RG_P36]",2.0,1.00,0.5,0.693426
4,K05,Known,"[RG_P21, RG_P22, RG_P17, RG_P23, RG_P08]",1.0,1.00,1.0,1.000000
5,K06,Known,"[RG_P23, RG_P22, RG_P17, RG_P24, RG_P18]",2.0,1.00,0.5,0.630930
6,K07,Known,"[RG_P24, RG_P25, RG_P26, RG_P15, MS_P01]",1.0,1.00,1.0,1.000000
7,K08,Known,"[RG_P28, RG_P23, RG_P22, RG_P29, RG_P34]",1.0,1.00,1.0,1.000000
8,I01,Inferred,"[RG_P25, RG_P26, MS_P01, RG_P23, RG_P13]",1.0,0.75,1.0,0.736590
9,I02,Inferred,"[RG_P23, RG_P22, RG_P17, RG_P18, RG_P25]",1.0,1.00,1.0,1.000000


## 2. Answer Generation

Convert the retrieved chunks into a structured context and pass this context to the language model.

The baseline uses a simple generation prompt because earlier testing showed that stricter grounding instructions caused the model to incorrectly refuse supported questions. Generated answers are stored alongside the retrieval results so later evaluation can trace each answer back to the exact evidence supplied to the model.

In [3]:
def build_context(row):
    context_parts = []

    for chunk_id, text in zip(
        row["retrieved_chunk_ids"],
        row["retrieved_contexts"]
    ):
        chunk = kb_df.loc[
            kb_df["chunk_id"] == chunk_id
        ].iloc[0]

        context_parts.append(
            f"[{chunk_id}] "
            f"{chunk['document_name']} - Page {chunk['page']}\n"
            f"{text}"
        )

    return "\n\n---\n\n".join(context_parts)


def generate_answer(question, context):
    prompt = f"""
Answer the question using the information in the context below.

Context:
{context}

Question:
{question}

Give a short answer and cite the supporting chunk ID.
"""

    response = requests.post(
        OLLAMA_URL,
        json={
            "model": OLLAMA_MODEL,
            "prompt": prompt,
            "stream": False,
            "options": {
                "temperature": 0
            }
        }
    )

    response.raise_for_status()

    return response.json()["response"].strip()


# Build contexts
results_df["retrieved_context"] = results_df.apply(
    build_context,
    axis=1
)

# Generate an answer for every test question
results_df["generated_answer"] = results_df.apply(
    lambda row: generate_answer(
        row["question"],
        row["retrieved_context"]
    ),
    axis=1
)

display(
    results_df[
        [
            "question_id",
            "question_type",
            "generated_answer"
        ]
    ]
)

,question_id,question_type,generated_answer
0,K01,Known,"No, a rental provider cannot ask you about dis..."
1,K02,Known,"No, a rental provider cannot accept an offer a..."
2,K03,Known,"Yes, a Victorian rental property must have a f..."
3,K04,Known,You have 5 business days to return your comple...
4,K05,Known,You can be asked to pay up to 2 weeks' rent in...
5,K06,Known,You must receive at least 90 days' notice befo...
6,K07,Known,"Yes, a broken toilet is considered an urgent r..."
7,K08,Known,If your rental provider wants to refuse your r...
8,I01,Inferred,A non-working fixed heater is considered a non...
9,I02,Inferred,You should receive a Notice of rent increase a...


## 3. Automatic Output Checks

Some RAG behaviours can be evaluated directly without using another language model as a judge.

This stage checks whether the generated answer contains citations, whether those citations refer to chunks that were actually retrieved, and whether the model abstained because it could not find enough information. These objective checks are kept separate from later semantic measures such as correctness, completeness and faithfulness.

In [4]:
# Extract cited chunk IDs from generated answers
def extract_citations(answer):
    return list(dict.fromkeys(
        re.findall(r"\b(?:RG|MS)_P\d{2}\b", answer)
    ))


# Detect common abstention responses
def detect_abstention(answer):
    text = answer.lower()

    abstention_phrases = [
        "does not contain enough information",
        "not enough information",
        "no mention of",
        "not mentioned in",
        "cannot determine from",
        "cannot be determined from",
        "context does not provide",
        "context does not contain"
    ]

    return int(any(phrase in text for phrase in abstention_phrases))


results_df["cited_chunk_ids"] = (
    results_df["generated_answer"]
    .apply(extract_citations)
)

results_df["citation_present"] = (
    results_df["cited_chunk_ids"]
    .apply(lambda x: int(len(x) > 0))
)

# Check that cited chunks were actually supplied to the model
results_df["citations_from_retrieved"] = results_df.apply(
    lambda row:
        int(
            all(
                citation in row["retrieved_chunk_ids"]
                for citation in row["cited_chunk_ids"]
            )
        )
        if row["cited_chunk_ids"]
        else np.nan,
    axis=1
)

results_df["abstained"] = (
    results_df["generated_answer"]
    .apply(detect_abstention)
)

# Ground-truth expectation for abstention
results_df["should_abstain"] = (
    results_df["question_type"] == "Out-of-KB"
).astype(int)

results_df["abstention_correct"] = (
    results_df["abstained"]
    == results_df["should_abstain"]
).astype(int)


display(
    results_df[
        [
            "question_id",
            "question_type",
            "cited_chunk_ids",
            "citation_present",
            "citations_from_retrieved",
            "abstained",
            "should_abstain",
            "abstention_correct"
        ]
    ]
)

,question_id,question_type,cited_chunk_ids,citation_present,citations_from_retrieved,abstained,should_abstain,abstention_correct
0,K01,Known,[RG_P07],1,1.0,0,0,1
1,K02,Known,[RG_P08],1,1.0,0,0,1
2,K03,Known,[RG_P39],1,1.0,0,0,1
3,K04,Known,[RG_P17],1,1.0,0,0,1
4,K05,Known,[],0,NaN,0,0,1
5,K06,Known,[RG_P24],1,1.0,0,0,1
6,K07,Known,[RG_P24],1,1.0,0,0,1
7,K08,Known,[RG_P28],1,1.0,0,0,1
8,I01,Inferred,[],0,NaN,0,0,1
9,I02,Inferred,[],0,NaN,0,0,1


### Automatic Evaluation Summary

Citation and abstention behaviour can be measured directly from the generated outputs. These checks provide objective system-level metrics before evaluating the meaning and factual support of individual answers.

In [5]:
# Summarise deterministic generation checks

answers_with_citations = results_df["citation_present"].mean()

valid_citations = results_df.loc[
    results_df["citation_present"] == 1,
    "citations_from_retrieved"
].mean()

out_of_kb = results_df[
    results_df["question_type"] == "Out-of-KB"
]

out_of_kb_abstention = out_of_kb["abstention_correct"].mean()

automatic_summary = pd.DataFrame({
    "metric": [
        "Citation presence",
        "Citations from retrieved context",
        "Out-of-KB abstention accuracy"
    ],
    "score": [
        answers_with_citations,
        valid_citations,
        out_of_kb_abstention
    ]
})

automatic_summary["score"] = automatic_summary["score"].round(3)

display(automatic_summary)

,metric,score
0,Citation presence,0.600
1,Citations from retrieved context,1.000
2,Out-of-KB abstention accuracy,0.333


## 4. Gold-Fact Evaluation

Each expected answer is broken into individual factual requirements. This allows answer completeness to be measured as the proportion of required facts included in the generated answer, rather than using a single binary complete/incomplete score.

These gold facts also make it easier to identify exactly which information the RAG system is missing.

In [6]:
# Required facts for each supported test question

gold_facts = {
    "K01": [
        "A rental provider or agent cannot ask whether the applicant has taken legal action or had a dispute with a previous rental provider."
    ],

    "K02": [
        "Rental providers or agents cannot accept an offer above the advertised rent."
    ],

    "K03": [
        "The property must have a fixed heater in good working order in the main living area.",
        "For rental agreements starting from 29 March 2023, the heater must be energy efficient."
    ],

    "K04": [
        "The completed and signed condition report must be returned within 5 business days of moving in."
    ],

    "K05": [
        "If rent is paid weekly, no more than 2 weeks rent can be requested in advance.",
        "If rent is paid monthly and weekly rent is $900 or less, no more than one month rent can be requested in advance.",
        "Different rules apply when weekly rent is above $900."
    ],

    "K06": [
        "A Notice of rent increase must be provided at least 90 days before the increase takes effect."
    ],

    "K07": [
        "A blocked or broken toilet system is an urgent repair.",
        "Urgent repairs must be dealt with immediately."
    ],

    "K08": [
        "A rental provider who wants to refuse a pet request must apply to VCAT within 14 days.",
        "VCAT decides whether refusing consent is reasonable."
    ],

    "I01": [
        "A rental property must have a working fixed heater in the main living area.",
        "Failure to meet rental minimum standards is considered an urgent repair.",
        "The rental provider must arrange an urgent repair immediately.",
        "If the provider does not respond, the renter can arrange the urgent repair if it costs no more than $2500.",
        "The rental provider must reimburse the renter within 7 days.",
        "If reimbursement is not provided, the renter can apply to RDRV."
    ],

    "I02": [
        "The renter must receive at least 90 days notice of a rent increase.",
        "The renter can request a free rent assessment from Consumer Affairs Victoria.",
        "The assessment must be requested within 30 days of receiving the rent increase notice.",
        "If the dispute remains unresolved, the renter can apply to RDRV or VCAT."
    ],

    "I03": [
        "The original condition report can show that the damage existed when the renter moved in.",
        "Photos can provide evidence of pre-existing damage.",
        "The exit condition report can be compared with the original condition report.",
        "The rental provider cannot claim the bond for fair wear and tear.",
        "A disputed bond claim can be taken through RDRV."
    ],

    "I04": [
        "A rental provider cannot evict a renter because they exercised or intended to exercise their rental rights.",
        "The rental provider needs a valid reason to end the agreement.",
        "The rental provider must provide the correct written Notice to vacate.",
        "The required notice period depends on the reason.",
        "In most instances the notice period is 90 days."
    ],

    "O01": [],
    "O02": [],
    "O03": []
}

results_df["gold_facts"] = (
    results_df["question_id"]
    .map(gold_facts)
)

results_df["gold_fact_count"] = (
    results_df["gold_facts"]
    .apply(len)
)

display(
    results_df[
        [
            "question_id",
            "question_type",
            "gold_fact_count",
            "gold_facts"
        ]
    ]
)

,question_id,question_type,gold_fact_count,gold_facts
0,K01,Known,1,[A rental provider or agent cannot ask whether...
1,K02,Known,1,[Rental providers or agents cannot accept an o...
2,K03,Known,2,[The property must have a fixed heater in good...
3,K04,Known,1,[The completed and signed condition report mus...
4,K05,Known,3,"[If rent is paid weekly, no more than 2 weeks ..."
5,K06,Known,1,[A Notice of rent increase must be provided at...
6,K07,Known,2,[A blocked or broken toilet system is an urgen...
7,K08,Known,2,[A rental provider who wants to refuse a pet r...
8,I01,Inferred,6,[A rental property must have a working fixed h...
9,I02,Inferred,4,[The renter must receive at least 90 days noti...


### Human Gold-Fact Evaluation

Answer completeness is evaluated at the individual fact level. Each required gold fact is manually checked against the generated answer and marked as either covered (`1`) or not covered (`0`).

Manual evaluation is used here because semantic coverage requires understanding paraphrased answers, and initial testing showed that the small local language model was not reliable enough to judge this consistently.

The final completeness score for each question is calculated automatically as the proportion of required facts covered by the generated answer.

In [11]:
# Create one evaluation row for each required gold fact

fact_rows = []

for _, row in results_df.iterrows():
    for fact_number, fact in enumerate(row["gold_facts"], start=1):
        fact_rows.append({
            "question_id": row["question_id"],
            "question_type": row["question_type"],
            "fact_number": fact_number,
            "gold_fact": fact,
            "generated_answer": row["generated_answer"],
            "covered": pd.NA,
            "review_notes": ""
        })

fact_eval_df = pd.DataFrame(fact_rows)

print("Total gold facts:", len(fact_eval_df))

display(fact_eval_df)

Total gold facts: 33


,question_id,question_type,fact_number,gold_fact,generated_answer,covered,review_notes
0,K01,Known,1,A rental provider or agent cannot ask whether ...,"No, a rental provider cannot ask you about dis...",<NA>,
1,K02,Known,1,Rental providers or agents cannot accept an of...,"No, a rental provider cannot accept an offer a...",<NA>,
2,K03,Known,1,The property must have a fixed heater in good ...,"Yes, a Victorian rental property must have a f...",<NA>,
3,K03,Known,2,For rental agreements starting from 29 March 2...,"Yes, a Victorian rental property must have a f...",<NA>,
4,K04,Known,1,The completed and signed condition report must...,You have 5 business days to return your comple...,<NA>,
5,K05,Known,1,"If rent is paid weekly, no more than 2 weeks r...",You can be asked to pay up to 2 weeks' rent in...,<NA>,
6,K05,Known,2,If rent is paid monthly and weekly rent is $90...,You can be asked to pay up to 2 weeks' rent in...,<NA>,
7,K05,Known,3,Different rules apply when weekly rent is abov...,You can be asked to pay up to 2 weeks' rent in...,<NA>,
8,K06,Known,1,A Notice of rent increase must be provided at ...,You must receive at least 90 days' notice befo...,<NA>,
9,K07,Known,1,A blocked or broken toilet system is an urgent...,"Yes, a broken toilet is considered an urgent r...",<NA>,


### Record Human Fact Judgements

Each gold fact is manually reviewed against the generated answer. A value of `1` means the required information is clearly present, while `0` means it is missing or contradicted.

These human labels provide the reference completeness evaluation for the baseline system.

In [12]:
# Human-reviewed gold fact coverage
# Format: (question_id, fact_number): covered

fact_coverage = {
    ("K01", 1): 1,

    ("K02", 1): 1,

    ("K03", 1): 1,
    ("K03", 2): 0,

    ("K04", 1): 1,

    ("K05", 1): 1,
    ("K05", 2): 1,
    ("K05", 3): 0,

    ("K06", 1): 1,

    ("K07", 1): 1,
    ("K07", 2): 0,

    ("K08", 1): 1,
    ("K08", 2): 1,

    ("I01", 1): 0,
    ("I01", 2): 0,
    ("I01", 3): 0,
    ("I01", 4): 1,
    ("I01", 5): 0,
    ("I01", 6): 0,

    ("I02", 1): 1,
    ("I02", 2): 1,
    ("I02", 3): 1,
    ("I02", 4): 0,

    ("I03", 1): 1,
    ("I03", 2): 1,
    ("I03", 3): 0,
    ("I03", 4): 0,
    ("I03", 5): 0,

    ("I04", 1): 1,
    ("I04", 2): 1,
    ("I04", 3): 1,
    ("I04", 4): 0,
    ("I04", 5): 1
}


# Apply human judgements
fact_eval_df["covered"] = fact_eval_df.apply(
    lambda row: fact_coverage.get(
        (row["question_id"], row["fact_number"]),
        pd.NA
    ),
    axis=1
)

# Check that every gold fact has been reviewed
print("Total gold facts:", len(fact_eval_df))
print("Reviewed facts:", fact_eval_df["covered"].notna().sum())
print("Missing judgements:", fact_eval_df["covered"].isna().sum())

display(
    fact_eval_df[
        [
            "question_id",
            "fact_number",
            "gold_fact",
            "covered"
        ]
    ]
)

Total gold facts: 33
Reviewed facts: 33
Missing judgements: 0


,question_id,fact_number,gold_fact,covered
0,K01,1,A rental provider or agent cannot ask whether ...,1
1,K02,1,Rental providers or agents cannot accept an of...,1
2,K03,1,The property must have a fixed heater in good ...,1
3,K03,2,For rental agreements starting from 29 March 2...,0
4,K04,1,The completed and signed condition report must...,1
5,K05,1,"If rent is paid weekly, no more than 2 weeks r...",1
6,K05,2,If rent is paid monthly and weekly rent is $90...,1
7,K05,3,Different rules apply when weekly rent is abov...,0
8,K06,1,A Notice of rent increase must be provided at ...,1
9,K07,1,A blocked or broken toilet system is an urgent...,1


### Completeness Results

Fact-level judgements are aggregated into a completeness score for each supported question.

A score of `1.0` means all required facts were included in the generated answer, while lower scores show that some expected information was missing. Results are also summarised by question type and across the full supported test set.

In [13]:
# Calculate completeness for each supported question

question_completeness = (
    fact_eval_df
    .groupby(["question_id", "question_type"])
    .agg(
        facts_covered=("covered", "sum"),
        total_facts=("covered", "count")
    )
    .reset_index()
)

question_completeness["completeness_score"] = (
    question_completeness["facts_covered"]
    / question_completeness["total_facts"]
)

# Add question-level completeness back to the main results
completeness_map = (
    question_completeness
    .set_index("question_id")["completeness_score"]
)

results_df["completeness_score"] = (
    results_df["question_id"]
    .map(completeness_map)
)


# Mean question-level completeness by type
completeness_by_type = (
    question_completeness
    .groupby("question_type")["completeness_score"]
    .mean()
    .round(3)
    .reset_index()
)

# Overall measures
overall_question_completeness = (
    question_completeness["completeness_score"].mean()
)

overall_fact_coverage = (
    fact_eval_df["covered"].sum()
    / len(fact_eval_df)
)

display(question_completeness)

display(completeness_by_type)

print(
    "Overall mean question completeness:",
    round(overall_question_completeness, 3)
)

print(
    "Overall gold-fact coverage:",
    round(overall_fact_coverage, 3)
)

,question_id,question_type,facts_covered,total_facts,completeness_score
0,I01,Inferred,1,6,0.166667
1,I02,Inferred,3,4,0.750000
2,I03,Inferred,2,5,0.400000
3,I04,Inferred,4,5,0.800000
4,K01,Known,1,1,1.000000
5,K02,Known,1,1,1.000000
6,K03,Known,1,2,0.500000
7,K04,Known,1,1,1.000000
8,K05,Known,2,3,0.666667
9,K06,Known,1,1,1.000000


,question_type,completeness_score
0,Inferred,0.529
1,Known,0.833


Overall mean question completeness: 0.732
Overall gold-fact coverage: 0.606


## 5. Claim-Level Faithfulness Evaluation

Faithfulness measures whether the factual claims made in each generated answer are supported by the context that was actually retrieved and provided to the model.

Generated answers are broken into individual factual claims. Each claim is manually checked against the retrieved context and marked as supported (`1`) or unsupported (`0`).

This allows the system to report both:

- **Faithfulness:** the proportion of generated claims supported by retrieved evidence.
- **Unsupported claim rate:** the proportion of generated claims that are not supported by the retrieved evidence.

This evaluation is based only on the retrieved context, not on whether a claim may be true according to outside knowledge.

In [14]:
# Manually identified factual claims from each generated answer

generated_claims = {
    "K01": [
        "A rental provider cannot ask about disputes with a previous rental provider when applying."
    ],

    "K02": [
        "A rental provider cannot accept an offer above the advertised rent."
    ],

    "K03": [
        "A Victorian rental property must have a fixed heater in good working order in the main living area."
    ],

    "K04": [
        "The completed condition report must be returned within 5 business days."
    ],

    "K05": [
        "If rent is paid weekly, up to 2 weeks rent can be requested in advance.",
        "If rent is paid monthly and weekly rent is $900 or less, up to one month rent can be requested in advance."
    ],

    "K06": [
        "At least 90 days notice must be given before a rent increase."
    ],

    "K07": [
        "A broken toilet is considered an urgent repair."
    ],

    "K08": [
        "A rental provider who wants to refuse a pet request must apply to VCAT within 14 days.",
        "VCAT decides whether refusing consent is reasonable."
    ],

    "I01": [
        "A non-working fixed heater is a non-urgent repair.",
        "If the rental provider does not respond, the renter can organise and pay for the repair if it costs no more than $2500.",
        "The renter can request reimbursement using the relevant notice form."
    ],

    "I02": [
        "At least 90 days notice must be provided before a rent increase takes effect.",
        "A renter can ask Consumer Affairs Victoria to investigate an excessive rent increase.",
        "The rent assessment request can be made within 30 days of receiving the rent increase notice."
    ],

    "I03": [
        "Pre-existing damage can be disputed using evidence from the original condition report.",
        "Photos can be used as evidence that damage existed before the renter moved in."
    ],

    "I04": [
        "A rental provider cannot evict a renter because they exercised their rental rights.",
        "A rental provider can only end the agreement for specific reasons.",
        "The rental provider must provide the required amount of notice.",
        "The required notice is usually 90 days.",
        "The notice must be given in the correct written form and include the reason."
    ],

    "O01": [
        "A renter cannot sublet a rental property without the rental provider's permission."
    ],

    "O02": [],

    "O03": [
        "A rental provider cannot install security cameras inside the rental property without the renter's consent."
    ]
}


claim_rows = []

for _, row in results_df.iterrows():
    for claim_number, claim in enumerate(
        generated_claims[row["question_id"]],
        start=1
    ):
        claim_rows.append({
            "question_id": row["question_id"],
            "question_type": row["question_type"],
            "claim_number": claim_number,
            "claim": claim,
            "retrieved_chunk_ids": row["retrieved_chunk_ids"],
            "supported": pd.NA,
            "supporting_chunk_ids": [],
            "review_notes": ""
        })

claim_eval_df = pd.DataFrame(claim_rows)

print("Total generated claims:", len(claim_eval_df))

display(claim_eval_df)

Total generated claims: 25


,question_id,question_type,claim_number,claim,retrieved_chunk_ids,supported,supporting_chunk_ids,review_notes
0,K01,Known,1,A rental provider cannot ask about disputes wi...,"[RG_P35, RG_P38, RG_P04, RG_P23, RG_P07]",<NA>,[],
1,K02,Known,1,A rental provider cannot accept an offer above...,"[RG_P08, RG_P21, MS_P01, RG_P12, RG_P13]",<NA>,[],
2,K03,Known,1,A Victorian rental property must have a fixed ...,"[RG_P13, MS_P01, RG_P39, RG_P17, RG_P04]",<NA>,[],
3,K04,Known,1,The completed condition report must be returne...,"[RG_P23, RG_P17, RG_P19, RG_P34, RG_P36]",<NA>,[],
4,K05,Known,1,"If rent is paid weekly, up to 2 weeks rent can...","[RG_P21, RG_P22, RG_P17, RG_P23, RG_P08]",<NA>,[],
5,K05,Known,2,If rent is paid monthly and weekly rent is $90...,"[RG_P21, RG_P22, RG_P17, RG_P23, RG_P08]",<NA>,[],
6,K06,Known,1,At least 90 days notice must be given before a...,"[RG_P23, RG_P22, RG_P17, RG_P24, RG_P18]",<NA>,[],
7,K07,Known,1,A broken toilet is considered an urgent repair.,"[RG_P24, RG_P25, RG_P26, RG_P15, MS_P01]",<NA>,[],
8,K08,Known,1,A rental provider who wants to refuse a pet re...,"[RG_P28, RG_P23, RG_P22, RG_P29, RG_P34]",<NA>,[],
9,K08,Known,2,VCAT decides whether refusing consent is reaso...,"[RG_P28, RG_P23, RG_P22, RG_P29, RG_P34]",<NA>,[],


### Human Claim Support Judgements

Each factual claim is manually checked against the retrieved context supplied to the language model.

A claim is marked as supported (`1`) only when the retrieved evidence clearly supports the claim as stated. Unsupported, contradicted or incorrectly applied claims are marked as `0`.

Supporting chunk IDs are also recorded so the source of each supported claim can be traced.

In [15]:
# Human-reviewed claim support
# Format:
# (question_id, claim_number): (supported, supporting_chunk_ids, review_note)

claim_support = {
    ("K01", 1): (1, ["RG_P07"], "Supported by the application rules."),

    ("K02", 1): (1, ["RG_P08"], "Supported by the rental bidding rules."),

    ("K03", 1): (1, ["RG_P13", "MS_P01"], "Both retrieved chunks support the fixed-heater requirement."),

    ("K04", 1): (1, ["RG_P17", "RG_P19"], "Supported by the condition report requirements."),

    ("K05", 1): (1, ["RG_P21"], "Supported by the weekly rent-in-advance rule."),
    ("K05", 2): (1, ["RG_P21"], "Supported by the monthly rent-in-advance rule."),

    ("K06", 1): (1, ["RG_P22"], "Supported by the 90-day rent increase notice rule."),

    ("K07", 1): (1, ["RG_P24"], "A broken toilet is explicitly listed as an urgent repair."),

    ("K08", 1): (1, ["RG_P28"], "Supported by the pet refusal process."),
    ("K08", 2): (1, ["RG_P28"], "Supported by the pet refusal process."),

    ("I01", 1): (
        0,
        [],
        "The retrieved evidence does not support classifying a non-working fixed heater as non-urgent."
    ),
    ("I01", 2): (
        0,
        [],
        "RG_P25 allows self-arranged repairs up to $2500 for urgent repairs, but the generated answer incorrectly applies this after classifying the heater issue as non-urgent."
    ),
    ("I01", 3): (
        0,
        [],
        "The reimbursement process is tied to the urgent-repair process, which the generated answer incorrectly applies to a repair it classified as non-urgent."
    ),

    ("I02", 1): (1, ["RG_P22"], "Supported by the rent increase notice rules."),
    ("I02", 2): (1, ["RG_P23"], "Supported by the rent assessment process."),
    ("I02", 3): (1, ["RG_P23"], "Supported by the 30-day assessment deadline."),

    ("I03", 1): (1, ["RG_P17"], "The condition report can document pre-existing damage."),
    ("I03", 2): (1, ["RG_P17"], "Photos can be used to document the property's condition."),

    ("I04", 1): (1, ["RG_P33"], "Supported by the protection against eviction for exercising rental rights."),
    ("I04", 2): (1, ["RG_P32", "RG_P33"], "Supported by the notice-to-vacate requirements."),
    ("I04", 3): (1, ["RG_P32"], "Supported by the required notice period."),
    ("I04", 4): (1, ["RG_P32"], "The guide states that the notice period is usually 90 days."),
    ("I04", 5): (1, ["RG_P32"], "Supported by the written Notice to vacate requirements."),

    ("O01", 1): (
        0,
        [],
        "None of the retrieved chunks contain information supporting the subletting claim."
    ),

    ("O03", 1): (
        0,
        [],
        "None of the retrieved chunks contain information supporting the security-camera claim."
    )
}


# Apply the human judgements
for index, row in claim_eval_df.iterrows():
    key = (row["question_id"], row["claim_number"])

    if key in claim_support:
        supported, chunks, note = claim_support[key]

        claim_eval_df.at[index, "supported"] = supported
        claim_eval_df.at[index, "supporting_chunk_ids"] = chunks
        claim_eval_df.at[index, "review_notes"] = note


print("Total generated claims:", len(claim_eval_df))
print("Reviewed claims:", claim_eval_df["supported"].notna().sum())
print("Missing judgements:", claim_eval_df["supported"].isna().sum())

display(
    claim_eval_df[
        [
            "question_id",
            "claim_number",
            "claim",
            "supported",
            "supporting_chunk_ids",
            "review_notes"
        ]
    ]
)

Total generated claims: 25
Reviewed claims: 25
Missing judgements: 0


,question_id,claim_number,claim,supported,supporting_chunk_ids,review_notes
0,K01,1,A rental provider cannot ask about disputes wi...,1,[RG_P07],Supported by the application rules.
1,K02,1,A rental provider cannot accept an offer above...,1,[RG_P08],Supported by the rental bidding rules.
2,K03,1,A Victorian rental property must have a fixed ...,1,"[RG_P13, MS_P01]",Both retrieved chunks support the fixed-heater...
3,K04,1,The completed condition report must be returne...,1,"[RG_P17, RG_P19]",Supported by the condition report requirements.
4,K05,1,"If rent is paid weekly, up to 2 weeks rent can...",1,[RG_P21],Supported by the weekly rent-in-advance rule.
5,K05,2,If rent is paid monthly and weekly rent is $90...,1,[RG_P21],Supported by the monthly rent-in-advance rule.
6,K06,1,At least 90 days notice must be given before a...,1,[RG_P22],Supported by the 90-day rent increase notice r...
7,K07,1,A broken toilet is considered an urgent repair.,1,[RG_P24],A broken toilet is explicitly listed as an urg...
8,K08,1,A rental provider who wants to refuse a pet re...,1,[RG_P28],Supported by the pet refusal process.
9,K08,2,VCAT decides whether refusing consent is reaso...,1,[RG_P28],Supported by the pet refusal process.


### Faithfulness Results

Claim-level support judgements are aggregated to measure how well the generated answers stay grounded in the retrieved evidence.

**Faithfulness** is the proportion of generated factual claims that are supported by the retrieved context. The **unsupported claim rate** is the proportion of claims that are unsupported or contradicted.

Questions where the model correctly abstains and makes no factual claims are not assigned a claim-level faithfulness score.

In [16]:
# Calculate faithfulness for each answer

question_faithfulness = (
    claim_eval_df
    .groupby(["question_id", "question_type"])
    .agg(
        supported_claims=("supported", "sum"),
        total_claims=("supported", "count")
    )
    .reset_index()
)

question_faithfulness["faithfulness_score"] = (
    question_faithfulness["supported_claims"]
    / question_faithfulness["total_claims"]
)

question_faithfulness["unsupported_claim_rate"] = (
    1 - question_faithfulness["faithfulness_score"]
)

# Add question-level faithfulness back to main results
faithfulness_map = (
    question_faithfulness
    .set_index("question_id")["faithfulness_score"]
)

results_df["faithfulness_score"] = (
    results_df["question_id"]
    .map(faithfulness_map)
)


# Claim-level results by question type
faithfulness_by_type = (
    claim_eval_df
    .groupby("question_type")
    .agg(
        supported_claims=("supported", "sum"),
        total_claims=("supported", "count")
    )
    .reset_index()
)

faithfulness_by_type["faithfulness_score"] = (
    faithfulness_by_type["supported_claims"]
    / faithfulness_by_type["total_claims"]
)

faithfulness_by_type["unsupported_claim_rate"] = (
    1 - faithfulness_by_type["faithfulness_score"]
)

faithfulness_by_type[
    ["faithfulness_score", "unsupported_claim_rate"]
] = faithfulness_by_type[
    ["faithfulness_score", "unsupported_claim_rate"]
].round(3)


# Overall claim-level results
overall_faithfulness = (
    claim_eval_df["supported"].sum()
    / len(claim_eval_df)
)

overall_unsupported_rate = 1 - overall_faithfulness


display(question_faithfulness)

display(faithfulness_by_type)

print(
    "Overall claim-level faithfulness:",
    round(overall_faithfulness, 3)
)

print(
    "Overall unsupported claim rate:",
    round(overall_unsupported_rate, 3)
)

,question_id,question_type,supported_claims,total_claims,faithfulness_score,unsupported_claim_rate
0,I01,Inferred,0,3,0.0,1.0
1,I02,Inferred,3,3,1.0,0.0
2,I03,Inferred,2,2,1.0,0.0
3,I04,Inferred,5,5,1.0,0.0
4,K01,Known,1,1,1.0,0.0
5,K02,Known,1,1,1.0,0.0
6,K03,Known,1,1,1.0,0.0
7,K04,Known,1,1,1.0,0.0
8,K05,Known,2,2,1.0,0.0
9,K06,Known,1,1,1.0,0.0


,question_type,supported_claims,total_claims,faithfulness_score,unsupported_claim_rate
0,Inferred,10,13,0.769231,0.230769
1,Known,10,10,1.0,0.0
2,Out-of-KB,0,2,0.0,1.0


Overall claim-level faithfulness: 0.8
Overall unsupported claim rate: 0.2


## 6. Semantic Citation Evaluation

Citation quality is evaluated separately from citation presence.

Two measures are used:

- **Citation precision:** whether the chunk IDs cited by the model actually support factual claims in the answer.
- **Citation coverage:** the proportion of supported factual claims that are backed by at least one citation in the generated answer.

This distinguishes between simply producing a valid-looking source ID and actually citing the correct evidence.

In [17]:
# Build the set of valid supporting chunks for each question

valid_supporting_chunks = (
    claim_eval_df[
        claim_eval_df["supported"] == 1
    ]
    .groupby("question_id")["supporting_chunk_ids"]
    .apply(
        lambda lists: set(
            chunk
            for chunk_list in lists
            for chunk in chunk_list
        )
    )
    .to_dict()
)


# Citation precision for each question
def calculate_citation_precision(row):
    cited = row["cited_chunk_ids"]

    if not cited:
        return np.nan

    valid_chunks = valid_supporting_chunks.get(
        row["question_id"],
        set()
    )

    correct_citations = sum(
        citation in valid_chunks
        for citation in cited
    )

    return correct_citations / len(cited)


results_df["citation_precision"] = results_df.apply(
    calculate_citation_precision,
    axis=1
)


# Check citation coverage at claim level
claim_eval_df["citation_covered"] = claim_eval_df.apply(
    lambda row:
        int(
            any(
                chunk in results_df.loc[
                    results_df["question_id"] == row["question_id"],
                    "cited_chunk_ids"
                ].iloc[0]
                for chunk in row["supporting_chunk_ids"]
            )
        )
        if row["supported"] == 1
        else np.nan,
    axis=1
)


# Citation coverage per question
citation_coverage = (
    claim_eval_df[
        claim_eval_df["supported"] == 1
    ]
    .groupby("question_id")["citation_covered"]
    .mean()
)

results_df["citation_coverage"] = (
    results_df["question_id"]
    .map(citation_coverage)
)


display(
    results_df[
        [
            "question_id",
            "question_type",
            "cited_chunk_ids",
            "citation_precision",
            "citation_coverage"
        ]
    ]
)


# Overall citation metrics
overall_citation_precision = (
    results_df.loc[
        results_df["citation_present"] == 1,
        "citation_precision"
    ].mean()
)

overall_citation_coverage = (
    claim_eval_df.loc[
        claim_eval_df["supported"] == 1,
        "citation_covered"
    ].mean()
)

print(
    "Overall semantic citation precision:",
    round(overall_citation_precision, 3)
)

print(
    "Overall supported-claim citation coverage:",
    round(overall_citation_coverage, 3)
)

,question_id,question_type,cited_chunk_ids,citation_precision,citation_coverage
0,K01,Known,[RG_P07],1.0,1.0
1,K02,Known,[RG_P08],1.0,1.0
2,K03,Known,[RG_P39],0.0,0.0
3,K04,Known,[RG_P17],1.0,1.0
4,K05,Known,[],NaN,0.0
5,K06,Known,[RG_P24],0.0,0.0
6,K07,Known,[RG_P24],1.0,1.0
7,K08,Known,[RG_P28],1.0,1.0
8,I01,Inferred,[],NaN,NaN
9,I02,Inferred,[],NaN,0.0


Overall semantic citation precision: 0.778
Overall supported-claim citation coverage: 0.6


## 7. Answer Correctness

Each generated response is manually reviewed for overall correctness.

A response is marked as correct (`1`) when its main answer is factually correct and does not contain a major contradiction. A response may still be marked as correct even if it is incomplete, since completeness is measured separately.

For Out-of-KB questions, correctness requires the system to recognise that the available evidence is insufficient rather than provide an unsupported answer.

In [18]:
# Human-reviewed overall answer correctness

answer_correctness = {
    "K01": 1,
    "K02": 1,
    "K03": 1,
    "K04": 1,
    "K05": 1,
    "K06": 1,
    "K07": 1,
    "K08": 1,

    "I01": 0,
    "I02": 1,
    "I03": 1,
    "I04": 1,

    "O01": 0,
    "O02": 1,
    "O03": 0
}

results_df["correct"] = (
    results_df["question_id"]
    .map(answer_correctness)
)

correctness_by_type = (
    results_df
    .groupby("question_type")["correct"]
    .mean()
    .round(3)
    .reset_index()
)

overall_correctness = results_df["correct"].mean()

display(
    results_df[
        [
            "question_id",
            "question_type",
            "correct",
            "completeness_score",
            "faithfulness_score"
        ]
    ]
)

display(correctness_by_type)

print(
    "Overall answer correctness:",
    round(overall_correctness, 3)
)

,question_id,question_type,correct,completeness_score,faithfulness_score
0,K01,Known,1,1.000000,1.0
1,K02,Known,1,1.000000,1.0
2,K03,Known,1,0.500000,1.0
3,K04,Known,1,1.000000,1.0
4,K05,Known,1,0.666667,1.0
5,K06,Known,1,1.000000,1.0
6,K07,Known,1,0.500000,1.0
7,K08,Known,1,1.000000,1.0
8,I01,Inferred,0,0.166667,0.0
9,I02,Inferred,1,0.750000,1.0


,question_type,correct
0,Inferred,0.750
1,Known,1.000
2,Out-of-KB,0.333


Overall answer correctness: 0.8


## 8. Baseline Evaluation Summary

The final baseline evaluation combines retrieval, answer quality, grounding, citation and abstention measures.

Each metric evaluates a different part of the RAG pipeline. Retrieval metrics are calculated only for questions with known relevant evidence, completeness is calculated for supported questions, faithfulness is measured at the generated-claim level, and abstention is evaluated on Out-of-KB questions.

This provides the Version 1 benchmark that later system improvements can be compared against using the same test collection and evaluation process.

In [19]:
# Consolidated Version 1 baseline metrics

supported_questions = results_df[
    results_df["question_type"].isin(["Known", "Inferred"])
]

summary_metrics = pd.DataFrame({
    "evaluation_area": [
        "Retrieval",
        "Retrieval",
        "Retrieval",
        "Answer quality",
        "Answer quality",
        "Answer quality",
        "Grounding",
        "Grounding",
        "Citations",
        "Citations",
        "Citations",
        "Citations",
        "Out-of-KB"
    ],

    "metric": [
        "Recall@5",
        "MRR@5",
        "NDCG@5",
        "Answer correctness",
        "Mean question completeness",
        "Gold-fact coverage",
        "Claim-level faithfulness",
        "Unsupported claim rate",
        "Citation presence",
        "Citations from retrieved context",
        "Semantic citation precision",
        "Supported-claim citation coverage",
        "Out-of-KB abstention accuracy"
    ],

    "score": [
        supported_questions["recall_at_k"].mean(),
        supported_questions["mrr_at_k"].mean(),
        supported_questions["ndcg_at_k"].mean(),

        results_df["correct"].mean(),
        question_completeness["completeness_score"].mean(),
        fact_eval_df["covered"].sum() / len(fact_eval_df),

        claim_eval_df["supported"].sum() / len(claim_eval_df),
        1 - (claim_eval_df["supported"].sum() / len(claim_eval_df)),

        results_df["citation_present"].mean(),

        results_df.loc[
            results_df["citation_present"] == 1,
            "citations_from_retrieved"
        ].mean(),

        results_df.loc[
            results_df["citation_present"] == 1,
            "citation_precision"
        ].mean(),

        claim_eval_df.loc[
            claim_eval_df["supported"] == 1,
            "citation_covered"
        ].mean(),

        results_df.loc[
            results_df["question_type"] == "Out-of-KB",
            "abstention_correct"
        ].mean()
    ]
})

summary_metrics["score"] = (
    summary_metrics["score"]
    .astype(float)
    .round(3)
)

display(summary_metrics)

,evaluation_area,metric,score
0,Retrieval,Recall@5,0.979
1,Retrieval,MRR@5,0.808
2,Retrieval,NDCG@5,0.831
3,Answer quality,Answer correctness,0.800
4,Answer quality,Mean question completeness,0.732
5,Answer quality,Gold-fact coverage,0.606
6,Grounding,Claim-level faithfulness,0.800
7,Grounding,Unsupported claim rate,0.200
8,Citations,Citation presence,0.600
9,Citations,Citations from retrieved context,1.000


## 9. Baseline Findings

The Version 1 RAG system performs well at retrieving relevant information, with **Recall@5 of 0.979**. This means the required evidence is usually present somewhere in the five chunks supplied to the language model. However, the lower **MRR@5 (0.808)** and **NDCG@5 (0.831)** show that relevant evidence is not always ranked near the top.

Final answer correctness was **0.800**, although answer completeness was lower. Mean question completeness was **0.732**, while only **60.6% of all required gold facts** were included. Known questions were generally handled well, while Inferred questions lost more information when multiple pieces of evidence needed to be combined.

Claim-level faithfulness was **0.800**, meaning 80% of factual claims made by the model were supported by the retrieved context. The remaining **20% were unsupported**, including incorrect claims in the heater example and unsupported answers to Out-of-KB questions.

Citation behaviour was mixed. Citations were included in only **60% of answers**. When citations were provided, semantic citation precision was **0.778**, showing that the model sometimes cited a retrieved chunk that did not actually support the claim. Only **60% of supported claims were accompanied by an appropriate citation**.

The clearest weakness was Out-of-KB handling. Abstention accuracy was only **0.333**, meaning the model correctly recognised insufficient evidence for only one of the three unsupported questions. In the other cases it produced confident answers without supporting evidence.

Overall, the baseline shows that retrieval is relatively strong, while the main opportunities for improvement are **multi-part answer completeness, grounding, citation behaviour and reliable abstention when evidence is missing**.

In [20]:
# Save Version 1 evaluation outputs

OUTPUT_DIR = DATA_DIR.parent / "evaluation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

results_df.to_csv(
    OUTPUT_DIR / "v1_question_results.csv",
    index=False
)

fact_eval_df.to_csv(
    OUTPUT_DIR / "v1_gold_fact_evaluation.csv",
    index=False
)

claim_eval_df.to_csv(
    OUTPUT_DIR / "v1_claim_evaluation.csv",
    index=False
)

summary_metrics.to_csv(
    OUTPUT_DIR / "v1_summary_metrics.csv",
    index=False
)

print("Saved Version 1 evaluation outputs to:")
print(OUTPUT_DIR)

Saved Version 1 evaluation outputs to:
/Users/seanrichards/Documents/University/DS Case Studies/Project/data/evaluation
